<p align="center">
  <img src="https://raw.githubusercontent.com/miniscope/minisim/main/docs/_static/logo/minisim_wordmark_sim.png" alt="Minisim" width="300">
</p>

# Build a miniscope recording, one stage at a time

A 1-photon miniscope recording is the end of a physical chain: neurons fire →
calcium binds an indicator → light is emitted → optics blur and dim it → the
brain moves → a sensor counts noisy photons. The minian *analysis* pipeline runs
that chain **backwards** to recover the neurons. Here we run it **forwards** —
and we build it **together**, choosing the physics as we go.

Each stage has the same rhythm:

1. **Understand** — what this piece of physics is, and what to watch.
2. **Explore** — drag sliders and *see* the effect.
3. **Commit** — set the values you want, add the stage, and move on.

The recording grows as we add stages; by the end we have a full synthetic movie
*and* the exact ground truth behind it — positions, traces, footprints, motion,
and which cells are even physically recoverable. That truth is what the next notebooks
use: Notebook 2 reads the biology back out by demixing, and Notebook 3 scores how well
a pipeline recovers it.

> The interactive sliders need a **live kernel** — run this notebook (don't just
> read it). Run cells top to bottom; each stage uses the choices committed above
> it.

## Setup

The simulator is `minisim`. We also pull in `mediapy` (smooth inline video playback) and a couple of tiny plotting helpers. `mediapy` finds `ffmpeg` on your PATH automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from ipywidgets import FloatSlider, HBox, IntSlider, VBox
from IPython.display import display
import mediapy

from minisim import (
    Acquisition, Optics, ImageSensor, Tissue, Output, Spec, simulate,
    PlaceNeurons, CellActivity, CellOptics, Composite,
    Neuropil, Vasculature, VesselLayer, Bleaching, BrainMotion, IlluminationProfile, Vignette, Leakage, Sensor,
)
from minisim.spec import order_steps  # canonical pipeline-ordering helper (not top-level)
# The plotting/interactive plumbing (inline video players, the slider dashboards,
# the GCaMP colour map) lives in minisim.notebooks._support so this notebook stays
# about the physics. It is teaching glue, not part of minisim's public API; open
# that module if you want to see how the dashboards are built.
from minisim.notebooks._support import (
    GCAMP, play, show, interactive_panel,
    build_dashboard_frames, build_components_frames, build_neuropil_frames,
)

# Interactive backend: each widget builds ONE figure and updates it in place as you
# drag a slider -- smooth (no flicker) and immune to VS Code's duplicate-output bug.
# If plots ever duplicate after reopening the notebook, run Command Palette ->
# "Developer: Reload Window".
%matplotlib widget
plt.ioff()  # don't auto-show figures; we display each canvas once, then mutate it
plt.rcParams["image.cmap"] = "magma"

## Our recording, as we build it

Two things carry forward as we go:

- **`acq`** — the *scope*: optics, sensor, tissue, frame rate, duration. We set
  this first (Stage 1).
- **`steps`** — the ordered list of construction stages. We start empty and
  **append one stage per section** as we commit to it.

`build()` simulates the recording from whatever stages we've committed so far;
`preview()` does the same on a small, fast FOV for responsive sliders. (`preview`
keeps the real pixel size and optics — so blur and brightness look *exactly*
right — and just images a smaller patch of tissue for a few seconds.)

In [ ]:
SEED = 7
steps = []          # filled via commit() below; the order you add stages does not matter

# acq is created in Stage 1; these helpers read whatever it currently is.
def build(until=None, extra=None):
    # full-resolution recording from the stages committed so far. `extra` is one
    # step to run on top of the committed ones without appending it (e.g. preview
    # the optics on the placed cells before optics is formally committed).
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=acq, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

def preview(extra=None, until=None, n_px=128, duration_s=3.0):
    # fast, small-FOV recording for slider exploration: same pixel size + optics,
    # just a smaller patch and fewer frames. `extra` is a step to try on top of
    # the committed ones without appending it.
    fast = acq.model_copy(update={
        "image_sensor": acq.image_sensor.model_copy(update={"n_px_height": n_px, "n_px_width": n_px}),
        "duration_s": duration_s,
    })
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=fast, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

def commit(*new):
    # Add or replace (by kind) committed steps. The order you commit stages in does
    # not matter -- simulate() always runs the pipeline in its one physically-correct
    # order (minisim.spec.order_steps) -- so you can revisit and retune any earlier stage,
    # then re-run, and nothing downstream breaks. `steps` is kept canonical so the
    # "stages committed so far" prints below read in execution order.
    kinds = {s.kind for s in new}
    steps[:] = order_steps([s for s in steps if s.kind not in kinds] + list(new))

> **You can retune any stage, then re-run.** Each stage below is committed with
> `commit(...)`, and the order you commit stages in does not matter: `simulate()`
> always runs the pipeline in its one physically-correct order (cells -> tissue ->
> motion -> sensor, via `minisim.order_steps`). So scroll back, change a slider or a
> parameter in an earlier stage, re-commit, and re-run the cells below -- the
> pipeline reorders itself and nothing breaks.

## Stage 1 — How miniscope imaging works

Before committing to a specific scope, let's build intuition for the physics that
turns a fluorescent neuron into pixels. **This section is a sandbox:** it runs the
simulator's real optics, tissue, and sensor math on a few hand-placed cells, but
it is *completely independent* of the recording we build later — drag anything you
like, nothing here carries forward until we **commit a scope** at the bottom.

The image is shown in **green**, the way GCaMP fluorescence looks: a dark field
with bright cells.

**Left — the side view** (the picture you'd sketch on a whiteboard): the miniscope
at top, its **working distance** down to the focal plane, the **tissue** we image
through, and five cells spanning a range of depths around the focal plane.
**Right — what the image sensor sees.**

First, the **cell type** — a setting at the top of the next cell, not a slider.
`_MORPHOLOGY` chooses the GCaMP variant: **soma-targeted** GCaMP (SomaGCaMP /
riboGCaMP) confines the indicator to the cell body, so each cell is a clean blob;
**cytosolic** GCaMP (the usual GCaMP6/7/8, and the default here) also fills the
**proximal dendrites**, adding a *random* number of thin, branched, fainter
projections, so no two cells look alike. Watch those dendrites as you add tissue
or push a cell off focus: the thin features are the *first* thing scatter, defocus,
and the noise floor erase, so usually only the in-focus, shallow cell keeps them,
which is exactly why resolving fine neurites through tissue is so hard. (Change
`MORPHOLOGY` and re-run to compare soma-targeted vs cytosolic.)

Then eight knobs, four families of physics:

- **Forming the image** — *NA* (light collection ∝ NA², so higher NA is mostly
  *brighter* here; its sharpening is tiny because diffraction is already
  *sub-pixel* — a miniscope is **pixel-limited**, not diffraction-limited),
  *magnification* (zooms the field of view) and *pixel pitch* (how coarsely the
  chip samples). These last two also set how much light lands on **each pixel**:
  a pixel collects flux over the object area it sees, `(pitch / magnification)²`,
  so higher mag *or* finer pitch spreads the same light over more pixels and dims
  every one of them — the **resolution-vs-SNR tradeoff**.
- **Depth & focus** — *tissue thickness* (how much tissue we image through: thin →
  bright & crisp, thick → dim & smeared by scatter + attenuation) and *focus* (the
  V4's tunable focus, **±150 µm**: a cell off the focal plane blurs, and one beyond
  reach can never be made sharp). The shaded band marks the **depth of field**,
  `≈ n·λ/NA²` — raise NA and watch it *narrow* (sharper but shallower focus). Note
  the asymmetry too: going *deeper* costs both focus *and* light, so deep cells
  fade fastest.
- **Field curvature** — *field curv*: a miniscope has no room for the lens elements
  that flatten the focal plane, so the in-focus surface is a shallow **bowl**, not a
  plane (the side view draws it): cells off the optical axis come into focus
  *shallower*, nearer the scope. The radius is large (real scopes ≈ 2–3 mm) so the
  bowl is gentle, but because the depth of field is only microns, even a small
  off-axis shift pushes edge cells out of focus — which is why the corners of a
  miniscope movie are softer and people keep their cells of interest centered. Dial
  the radius down to exaggerate it.
- **Detecting the image** — *exposure* and *read noise*: the sensor turns light
  into noisy counts, and a dim cell can sink below the noise floor. That "is it
  above the noise?" question is exactly what decides whether the analysis pipeline
  can ever recover a cell.

In [ ]:
from _anatomy_panels import ImagingSandbox

# Cell type (edit + re-run to switch). Soma-targeted GCaMP confines signal to the
# body; cytosolic (the default) also fills a random number of branched dendrites -- the
# thin features that scatter, defocus, and the noise floor erase first. The scope
# schematic and the five-cell optics demo live in the sibling _anatomy_panels.py;
# what stays here is what you actually turn -- the cell type and the eight knobs.
MORPHOLOGY = "cytosolic"   # "soma" or "cytosolic"
DENDRITE_LEN_UM = 24.0     # nominal proximal reach, um (per-cell count is random)
DENDRITE_WIDTH_UM = 3.0    # base width (tapers to a thread toward the tip)

# Eight knobs, four families of physics (see the text above): forming the image
# (NA / magnification / pixel pitch), depth & focus (tissue thickness / focus
# offset), field curvature, and detecting the image (exposure / read noise).
sliders = dict(
    na=FloatSlider(min=0.1, max=0.6, step=0.02, value=0.30, description="NA"),
    magnification=FloatSlider(min=1.0, max=10.0, step=0.5, value=4.0, description="mag"),
    pixel_pitch_um=FloatSlider(min=1.0, max=8.0, step=0.2, value=5.0, description="pitch um"),
    tissue_thickness_um=FloatSlider(min=0.0, max=600.0, step=20.0, value=100.0, description="tissue um"),
    focus_offset_um=FloatSlider(min=-150.0, max=150.0, step=10.0, value=0.0, description="focus um"),
    exposure=FloatSlider(min=1000.0, max=20000.0, step=500.0, value=6000.0, description="exposure"),
    read_noise_e=FloatSlider(min=0.0, max=10.0, step=0.5, value=2.0, description="read noise"),
    field_curv_mm=FloatSlider(min=0.5, max=8.0, step=0.5, value=2.5, description="field curv mm"),
)

sandbox = ImagingSandbox(sliders, morphology=MORPHOLOGY,
                         dendrite_len_um=DENDRITE_LEN_UM, dendrite_width_um=DENDRITE_WIDTH_UM)
interactive_panel(sliders, sandbox.draw, sandbox.canvas)

**Commit the scope — the UCLA Miniscope V4.** Everything above was
intuition-building and is now behind us. Here we fix the *actual* scope the rest of
the notebook uses, independent of whatever you dragged in the sandbox. These are V4
numbers: a **Python480 CMOS** sensor at **608×608**, 4.8 µm pixels behind a ~3×
GRIN objective (**effective NA 0.30**) → a ~0.97 mm field at 1.6 µm/px, imaging
GCaMP, with a ~700 µm front working distance and ~**2.1 mm** field curvature. The
depth of field isn't a free knob — it's *derived from NA* (`"auto"`), since the
optics set it. The **focus** is also left `"auto"`: the V4's focus is tunable, so
rather than pin a fixed depth we let it settle where it **minimizes total defocus**
over the layer we place next — and because that calculation folds in the field
curvature (off-axis cells focus shallower), it lands a little deeper than the naive
middle of the layer, recovering more of the off-axis cells. Exactly what you do at
the rig: turn the focus knob until the most cells are sharp. (Once the sensor lands
in a later stage, `"auto"` upgrades from sharpest-average to *most-recoverable*: the
plane that maximizes how many cells clear the noise floor; we record it either way
as `focal_depth_um`.)

In [ ]:
# UCLA Miniscope V4. Two "auto" fields: depth of field is derived from NA
# (n*lambda/NA^2), and the focus tracks the placed layer (curvature-aware) -- with no
# sensor yet it minimizes total defocus (median effective depth); once the sensor is
# committed it maximizes recoverable yield. The V4 focus is tunable, so we don't pin
# it. front_working_distance_um is the V4 lens->focus spec (informational).
acq = Acquisition(
    fps=20.0,
    duration_s=90.0,
    focal_depth_in_tissue_um="auto",       # tunable V4 focus -> tracks placed layer
    front_working_distance_um=700.0,       # V4 spec; does not affect the simulation
    optics=Optics(
        na=0.30,                           # V4 effective NA
        magnification=3.0,
        emission_nm=525.0,                 # GCaMP emission
        field_curvature_radius_um=2100.0,  # V4 Petzval radius ~2.1 mm (no field flattener)
    ),
    image_sensor=ImageSensor(n_px_height=608, n_px_width=608,  # Python480 CMOS
                             pixel_pitch_um=4.8, bit_depth=8),
    tissue=Tissue(scatter_mfp_excitation_um=600.0,  # 470 nm in: diffuse, penetrates far
                  scatter_mfp_emission_um=100.0,    # 525 nm out: image-forming scattering MFP
                  scatter_blur_per_um=0.05),
)
print(f"FOV {acq.fov_um[0]:.0f} x {acq.fov_um[1]:.0f} um | {acq.pixel_size_um:.2f} um/px "
      f"| DOF +/-{acq.optics.resolved_depth_of_field_um:.1f} um | {acq.n_frames} frames "
      f"@ {acq.fps:.0f} fps | focus: {acq.focal_depth_in_tissue_um}")

## Stage 2 — Place the neurons

Now we drop cell bodies into the tissue volume: a lateral position `(x, y)` plus a
**depth** `z`. We don't choose *how many* directly — the count falls out of the
**volumetric density** (cells/mm³) and the imaged volume
(`round(density · FOV-area · thickness)`), so a thicker slab holds proportionally
more cells, just like a real chunk of tissue. (A strictly planar layer would have
zero thickness, so we floor it at one soma diameter — a single sheet of cell bodies
still has cells.)

Depth is the quiet protagonist of this whole notebook: it decides how blurred and
how dim each cell becomes, and ultimately whether it is recoverable at all. So this
section has two beats. **First the placement itself**, shown sharp and optics-free —
the true population in the tissue. **Then, at the end, we push that exact placement
through the V4 optics** to see what the scope makes of it (`A_planted → A_observed`).
That degradation is *purely spatial* — it depends only on each cell's depth and the
focus, not on any calcium activity — so we can apply it the moment the cells exist.

Start from a **preset**, then fine-tune:

- **CA1** — a thin pyramidal layer (~150 µm deep, near a single cell sheet).
- **Volumetric** — a thick 50–200 µm slab, with cells at every depth.

You also pick the **GCaMP variant** — *soma-targeted* (cell body only) or
*cytosolic* (soma + a few proximal dendrites). It does not move the distribution
dots (those mark cell bodies), but you will see it in the `A_planted → A_observed`
panels below: cytosolic's thin dendrites are precisely what the optics erase first.

Two views of the same population, at the **full committed FOV**:

- **Top** — looking straight down, each neuron a disk at its true soma size,
  **colored by depth** (the third placement axis). At realistic density the cell
  *bodies* stay mostly well separated — hundreds of cells the analysis must tell
  apart, with the occasional close pair that becomes a hard case. (Real crowding,
  where neighbors smear into one another, comes from the **optics** — which you see
  at the end of this section.)
- **Side** — the same lateral `x`, now against depth `z`: the slab the cells
  occupy, the structure the flat top view hides.

**Explore:** density, the depth band, soma size, and `min_distance` — a Poisson-disk
spacing floor. Raise it and the layout regularizes; high enough and the count
**caps out**, since you cannot pack past the spacing. This stage is **purely
spatial** — just where the cells sit. How brightly each one responds is biology we
set in Stage 3, and the noise that turns brightness into a signal-to-noise ratio
comes later still, from the optics and the sensor. Whatever you land on here is what
we commit and image below, so this is also where you come back to retune once you
have seen the finished recording.

In [ ]:
from ipywidgets import ToggleButtons
from _anatomy_panels import PlacementView

# Stage 2 runs at the FULL committed FOV (no preview shrink) -- seeing the real field
# is the whole point. PlacementView samples positions with sample_neurons (the placement
# half of place_neurons, no footprints) so even hundreds of cells redraw instantly; the
# top/side scatter is drawn by minisim.notebooks._support.plot_population.
_sliders2 = dict(
    density_per_mm3=FloatSlider(min=2000, max=60000, step=1000, value=45000, description="density/mm3"),
    depth_lo=FloatSlider(min=0, max=250, step=5, value=140, description="depth lo um"),
    depth_hi=FloatSlider(min=0, max=300, step=5, value=160, description="depth hi um"),
    soma_radius_um=FloatSlider(min=2, max=12, step=0.5, value=5, description="soma r um"),
    min_distance_um=FloatSlider(min=0, max=40, step=2, value=0, description="min dist um"),
)

# Presets jump the sliders to a scenario; fine-tune from there. CA1 = a thin pyramidal
# sheet; Volumetric = a thick slab spanning many depths. (The defaults match CA1.)
_PRESETS2 = {
    "CA1 (thin ~150um)":     dict(density_per_mm3=45000, depth_lo=140, depth_hi=160, soma_radius_um=5, min_distance_um=0),
    "Volumetric (50-200um)": dict(density_per_mm3=8000, depth_lo=50, depth_hi=200, soma_radius_um=6, min_distance_um=0),
}
_preset2 = ToggleButtons(options=list(_PRESETS2), value="CA1 (thin ~150um)", description="preset")


def _apply_preset2(change):
    for _k, _val in _PRESETS2[change["new"]].items():
        _sliders2[_k].value = _val


_preset2.observe(_apply_preset2, names="value")

# GCaMP variant: doesn't move the dots (they mark cell bodies); it shows up in the
# A_planted -> A_observed panels below, where cytosolic's dendrites erase first.
_morph2 = ToggleButtons(options=["soma", "cytosolic"], value="cytosolic", description="GCaMP")

_placement = PlacementView(_sliders2, _morph2, fov_um=acq.fov_um, seed=SEED)
_morph2.observe(lambda _c: _placement.draw(), names="value")  # redraw to update the title
display(HBox([_preset2, _morph2]))
interactive_panel(_sliders2, _placement.draw, _placement.canvas)

**Commit the placement.** We read the sliders above, so whatever you dialed in (preset + tweaks) is exactly what gets imaged. Re-running is idempotent — it *replaces* any previous placement rather than stacking a second one — so retune and re-run this as often as you like.

In [ ]:
_v2 = {k: s.value for k, s in _sliders2.items()}
commit(PlaceNeurons(
    density_per_mm3=_v2["density_per_mm3"], soma_radius_um=_v2["soma_radius_um"],
    depth_range_um=(_v2["depth_lo"], max(_v2["depth_hi"], _v2["depth_lo"])),
    min_distance_um=_v2["min_distance_um"], morphology=_morph2.value))
print(f"committed placement ({_morph2.value} GCaMP):", {k: round(float(x), 1) for k, x in _v2.items()})

### See it through the scope — `A_planted → A_observed`

The optical degradation is **purely spatial**: it depends on each cell's depth and
the focus, not on its calcium activity, so we apply it the moment the cells are
placed. Every sharp planted footprint is **dimmed** — light scatters and is absorbed
climbing back out of the tissue, and only the fraction set by NA² is collected — and
**blurred** by diffraction, by defocus for any cell off the focal surface, and by
scatter. The result, `A_observed`, is the best the analysis could ever recover; the
sharp `A_planted` is the answer key it will be graded against.

We focus on the layer we placed (`focal_depth_in_tissue_um="auto"` → the depth that
minimizes total defocus, which folds in the curvature below), just like turning the
focus knob at the rig. Two things then shape what survives: **depth** (cells far
from the focal depth blur and dim) and **field curvature** (the V4 has no
field-flattener, so its focal surface bows *shallower* toward the edges — even a
cell at exactly the right depth falls out of focus away from the optical axis; the
`"auto"` focus already leans deeper to claw some of them back). So for a **thin CA1
layer**, where depth is uniform, the
sharp cells form a central island that softens toward the rim — that is pure field
curvature, and it is why people keep their cells of interest near center. For the
**volumetric slab**, depth and curvature compound: only mid-depth, near-axis cells
stay crisp, shallow ones blur, and the deep ones fade toward nothing. Switch the
preset above, re-run the commit, and re-run this cell to compare — that is the
retuning loop in action.

> Each panel is scaled to its own peak with the GCaMP green LUT, so the faint
> observed field is lifted to full visibility: the top row is about *shape* (which
> cells stay sharp, which smear out), not brightness. The real dimming, observed
> peaks at only a few percent of planted, is reported in the title and bites later
> at the sensor noise floor. For the volumetric slab you can still read *which*
> cells stay legible, the sharp mid-depth near-axis ones. The bottom pair zooms a single cell,
> also per-panel normalized, isolating the **blur** the optics adds on top.

In [ ]:
from _anatomy_panels import optics_reveal_figure

# Degradation is independent of calcium, so push the committed placement straight
# through the optics (place_neurons + optics; focus "auto" tracks the layer). This is
# the one slow build in Stage 2 -- it stamps and degrades a footprint per cell. The
# A_planted -> A_observed reveal (shape vs brightness, the resolved focus, the in-focus
# count) is assembled in _anatomy_panels.optics_reveal_figure.
rec2 = build(until="optics", extra=CellOptics())
_figO = optics_reveal_figure(rec2.ground_truth, px_um=acq.pixel_size_um,
                             dof_um=acq.optics.resolved_depth_of_field_um, morph_label=_morph2.value)
show(_figO)

## Stage 3 — Calcium activity

Placement was about *where* cells are; this stage is about *when they fire* and the
fluorescence that follows. Each cell gets a spike train from a 2-state
(quiescent/active) gate driving a Poisson process, and every spike adds a calcium
transient with a fast rise and slow decay, `k(t) = e^{-t/τ_d} − e^{-t/τ_r}`. That
rise/decay shape is exactly what deconvolution later assumes.

Spikes are generated on a fine **~300 Hz** grid (one spike per ~3 ms bin respects the
refractory period), convolved with the kernel at that rate, then **binned down to the
camera frame rate** — exactly what the sensor's exposure does. So bursts can be dense
and physically tight even though we only ever store spikes at the frame rate.

Amplitude here is **biology**: a per-cell **brightness** gain (expression / response
strength) that scales a cell's *whole* trace, so some cells are simply brighter than
others. What is deliberately *not* here is **noise**: the trace you see is the clean
ground-truth `C`. Measurement noise is something the physical world adds later,
photon shot noise and read noise at the **sensor**, diffuse background at
**neuropil**, so a cell's signal-to-noise ratio is an *outcome* of the whole chain,
never a knob you set on the biology.

**Explore:**
- **peak t / FWHM** — the kernel's shape, in features you can read straight off a
  transient (time-to-peak, full width at half max); the simulator deduces `τ_r`/`τ_d`.
- **activity** — one sparse→dense knob (CaLab's spike-activity axis). Turning it up
  makes bursts start more often, last longer, and fire harder all at once, plus a
  little more background firing between them. 0.5 is CaLab's "moderate" default.
- **bright spread** — cell-to-cell brightness heterogeneity (a few bright cells over
  a dimmer majority).

(The sampling rule `τ_d · fps ≳ 1` must hold or the decay is unresolvable.)

In [ ]:
from minisim.steps import spike_activity_params, tau_from_kernel_timing
from _anatomy_panels import ActivityPanel

# Kernel shape is set by observable features (time-to-peak, FWHM); the simulator inverts
# them to tau_r/tau_d. Defaults ~ GCaMP6f (tau_r~50ms, tau_d~0.5s). The three panels
# (traces+spikes, kernel, brightness spread) are drawn by _anatomy_panels.ActivityPanel.
_sliders3 = dict(
    kernel_peak=FloatSlider(min=0.04, max=0.40, step=0.01, value=0.13, description="peak t (s)"),
    kernel_fwhm=FloatSlider(min=0.15, max=1.50, step=0.05, value=0.50, description="FWHM (s)"),
    activity=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description="activity"),
    brightness_cv=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.4, description="bright spread"),
)


def _activity_from_sliders(v):
    # observable kernel features -> tau_r/tau_d; one "activity" knob (sparse->dense)
    # moves all four CaLab Markov params together (see spike_activity_params).
    tau_r, tau_d = tau_from_kernel_timing(v["kernel_peak"], v["kernel_fwhm"])
    p_q2a, p_a2q, active_rate, quiescent_rate = spike_activity_params(v["activity"])
    return CellActivity(active_rate_hz=active_rate, quiescent_rate_hz=quiescent_rate,
                        tau_rise_s=tau_r, tau_decay_s=tau_d,
                        p_quiescent_to_active=p_q2a, p_active_to_quiescent=p_a2q,
                        brightness_cv=v["brightness_cv"]), tau_r, tau_d


_activity = ActivityPanel(_sliders3, preview, _activity_from_sliders)
interactive_panel(_sliders3, _activity.draw, _activity.canvas)

**Commit the activity.** Reads the sliders above, so the kernel shape, firing/burstiness, and brightness you dialed in are exactly what gets rendered. Idempotent: re-running *replaces* any prior activity.

In [ ]:
_v3 = {k: s.value for k, s in _sliders3.items()}
_step3, _tau_r3, _tau_d3 = _activity_from_sliders(_v3)
commit(_step3)
print(f"committed activity: kernel tau_r {_tau_r3*1000:.0f}ms / tau_d {_tau_d3*1000:.0f}ms, "
      f"activity {_v3['activity']} (0=sparse, 0.5=moderate, 1=dense), "
      f"brightness_cv {_v3['brightness_cv']}")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

---

### Next: the cells become a movie

So far we have cells with positions, depths, and traces — but no image yet. The
next stages turn them into a movie and then degrade it the way the physical world
does: **optics** (blur + dim by depth), **composite** (the first actual video!),
then **background, bleaching, motion, illumination, and sensor noise**. Those
stages produce real video at full resolution, so we'll decide *there* how to
balance slider responsiveness against fidelity — stage by stage.

## Stage 4 - Composite the movie

We finally have everything a frame needs: **where** each cell is and how the optics
degrade it (Stage 2, the `A_planted -> A_observed` reveal), and **when** it fires
(Stage 3, the trace `C`). Compositing just puts them together. Every cell contributes
its observed footprint, scaled frame by frame by its own calcium trace, and the
frame is the sum over all cells:

$$\text{movie}[t] \;=\; \sum_i A^{\text{obs}}_i \cdot C_i[t]$$

That product is the whole point of the module. A calcium movie is a low-rank
object: a handful of fixed spatial footprints `A`, each breathing in time by its
trace `C`. **CNMF, the analysis pipeline, exists to invert exactly this** - to
recover `A` and `C` from the movie. Here we run it forward, so we hold the true
`A` and `C` that any analysis is trying to get back.

This is the **first real video**, and it is deliberately *clean*: no background
haze, no bleaching, no motion, no sensor noise yet. Those degrade it in the stages
that follow, so this is the best-case image the pipeline could ever be handed.

We render a ~20 s clip and play it as a single video (smooth in the browser, unlike
a live slider). To make the `A.C` idea concrete, **five cells are picked out in
unique colours**: each is ringed in its colour in the movie, and shown on the right
as its footprint thumbnail next to its trace `C`. A vertical cursor sweeps the
traces in step with the video, so you can watch a ring brighten exactly as its
trace spikes.

In [ ]:
# Render ONCE over a ~20 s window, then play it as one video. We apply the optics
# previewed in Stage 2 (CellOptics) + the composite (Composite) on top of the committed
# stages without committing them yet -- the commit cell below does that.
_WINDOW_S = 20.0
_acq4 = acq.model_copy(update={"duration_s": _WINDOW_S})
_rec4 = simulate(Spec(acquisition=_acq4, seed=SEED, output=Output(save_intermediates=True),
                      steps=list(steps) + [CellOptics(), Composite()]), until="cells_only")
_gt4 = _rec4.ground_truth
_mov4 = np.asarray(_rec4.observed, dtype=np.float32)   # keep float32 (~0.6 GB at 20 s full-res)
_px4 = _rec4.spec.acquisition.pixel_size_um
_t4 = np.arange(_mov4.shape[0]) / _rec4.spec.acquisition.fps
_vmax4 = float(np.percentile(_mov4, 99.9))

# Pick ~5 busy, detectable, spatially separated cells to colour-code.
_colors4 = ["#ff5050", "#3fa7ff", "#ffd23f", "#ff5dff", "#ff9a3f"]
_order4 = np.argsort(_gt4.S.sum(axis=1))[::-1]
_det4 = [int(u) for u in _order4 if _gt4.detectable[u]] or [int(u) for u in _order4]
_picks4 = []
for _u in _det4:
    _cy, _cx = _gt4.centers_um[_u, 1] / _px4, _gt4.centers_um[_u, 2] / _px4
    if all(np.hypot(_cy - _gt4.centers_um[p, 1] / _px4, _cx - _gt4.centers_um[p, 2] / _px4) > 90.0
           for p in _picks4):
        _picks4.append(_u)
    if len(_picks4) == 5:
        break

_frames4 = build_dashboard_frames(_mov4, _gt4, _picks4, _colors4, _t4, _vmax4, _px4)
play(_frames4, fps=_rec4.spec.acquisition.fps,
     title="rendered movie: 5 cells colour-coded with their traces")

**Commit the optics + composite.** Locks in the depth blur/dimming previewed in Stage 2 and the movie compositing. Idempotent: re-running replaces any prior optics/composite so the stage list never accumulates duplicates.

In [ ]:
commit(CellOptics(), Composite())
print("committed: optics (depth blur + dimming) -> composite (the movie)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

## Stage 5 - Neuropil (the diffuse background haze)

Stage 4 gave us the clean movie $\text{movie}[t]=\sum_i A_i\,C_i[t]$. Real recordings
also carry a smooth, glowing **background**, and the analysis model is really

$$\text{movie}[t] \;=\; \underbrace{\sum_i A_i\,C_i[t]}_{\text{cells } (A\cdot C)} \;+\; \underbrace{\sum_k S_k\,T_k[t]}_{\text{background } (b\cdot f)}$$

The key idea: **the background is low-rank too.** It is a handful of smooth spatial
fields $S_k$, each breathing in time by its own envelope $T_k$ - exactly the same
$A\cdot C$ form as the cells, just diffuse and blurry instead of focal. CNMF's
background term `b·f` is precisely this, and here we build it forward.

What makes it **biology**, not just a few arbitrary components: the haze is the
dendritic and axonal felt of the surrounding neurons, so its time course is driven
by the **local population activity** $P(t)$ (the aggregate calcium, lagged and
smoothed by the felt's integration). Each $T_k$ mixes that shared population driver
with an independent slow drift (the unmodeled out-of-FOV tissue), set by
`population_coupling`. So the background brightens when the population fires.

(This is the modeled diffuse mesh *only*. Out-of-focus somata are a separate
background that already emerges for free from `place_neurons` + `optics`.)

**Where do the spatial fields $S_k$ come from?** Not from the cells. Modeling the
true mesh - the tangle of dendritic and axonal projections winding through the
volume - is genuinely hard, so we don't: each $S_k$ is **low-pass-filtered random
noise** (smooth blobs on a `spatial_sigma_um` scale), not derived from any neuron's
position. Only the *time course* $T_k$ is biology-driven. How many fields there are
is set by `n_components` (default **3**).

First, the background's own components - the three $S_k$ fields and their $T_k$
traces, in the same dashboard layout as the cells. We map the three components to
the **red, green, and blue channels**, so the movie on the left is their true-colour
sum: you can read each component's extent by its hue and watch a channel brighten
as its $T_k$ rises (where two overlap the colours mix). $P(t)$ rides on top so you
can see every component tracking it:

In [ ]:
# Render the committed scope over a ~20 s window both WITHOUT and WITH neuropil,
# so we can show the background's components and then the result of adding them in.
_WINDOW_S5 = 20.0
_acq5 = acq.model_copy(update={"duration_s": _WINDOW_S5})
_base5 = list(steps)   # committed so far: place -> activity -> optics -> composite
_rec5c = simulate(Spec(acquisition=_acq5, seed=SEED, output=Output(save_intermediates=True),
                       steps=_base5), until="cells_only")
_rec5b = simulate(Spec(acquisition=_acq5, seed=SEED, output=Output(save_intermediates=True),
                       steps=_base5 + [Neuropil()]), until="neuropil")
_gt5 = _rec5b.ground_truth
_clean5 = np.asarray(_rec5c.observed, dtype=np.float32)
_withbg5 = np.asarray(_rec5b.observed, dtype=np.float32)
_t5 = np.arange(_clean5.shape[0]) / _acq5.fps
_px5 = _acq5.pixel_size_um
_vmax5 = float(np.percentile(_withbg5, 99.9))

# Video 1: the background's own A.C -- 3 components mapped to R/G/B channels, so
# the left movie's channel k = S_k . T_k[t] and the colours sum to the haze.
_cframes5 = build_components_frames(_gt5.neuropil_spatial, _gt5.neuropil_temporal,
                                    _gt5.neuropil_population, _t5)
play(_cframes5, fps=_acq5.fps,
     title="background components in R/G/B: each channel is a field S_k breathing by T_k")

Now **add it onto the composite.** Same five cells as Stage 4, clean movie on the
left and clean+haze on the right. The traces on the far right need an honest label.
**They are not the true calcium $C$.** Each is a *naive footprint-ROI trace*: we
average the rendered movie over the cell's footprint, frame by frame - exactly what
you get if you draw a region of interest and read out its mean, with no unmixing.
We plot two, on the same scale: the coloured one is that ROI on the **clean** movie,
the grey one on the **clean + neuropil** movie.

Two contaminations are baked into this ROI, and they are the whole reason
**demixing** exists:

- **The neuropil pedestal** - the gap between the grey and coloured traces. It rises
  and falls together across all five cells because they share the population driver
  $P(t)$; a shared, additive background a subtraction step must peel off.
- **Neighbour bleed** - even the *coloured* (no-haze) ROI is not the cell's own
  signal. Its footprint also collects light from nearby cells that the optics blur
  and the tissue scatters into it. Under 1p imaging that bleed is large.

The true $C$ - which we hold in ground truth - can only be recovered by separating
each cell's contribution from its neighbours' and from the background. That is
CNMF's $A\cdot C + b\cdot f$ factorization (Stage 4, run in reverse), and we give it
its own stage at the end of the notebook.

In [ ]:
# Video 2: the "add it in" reveal -- clean | clean+neuropil + trace contamination.
_cellcolors5 = ["#ff5050", "#3fa7ff", "#ffd23f", "#ff5dff", "#ff9a3f"]
_order5 = np.argsort(_gt5.S.sum(axis=1))[::-1]
_det5 = [int(u) for u in _order5 if _gt5.detectable[u]] or [int(u) for u in _order5]
_picks5 = []
for _u in _det5:
    _cy, _cx = _gt5.centers_um[_u, 1] / _px5, _gt5.centers_um[_u, 2] / _px5
    if all(np.hypot(_cy - _gt5.centers_um[p, 1] / _px5, _cx - _gt5.centers_um[p, 2] / _px5) > 90.0
           for p in _picks5):
        _picks5.append(_u)
    if len(_picks5) == 5:
        break

_frames5 = build_neuropil_frames(_clean5, _withbg5, _gt5, _picks5, _cellcolors5, _t5, _vmax5, _px5)
play(_frames5, fps=_acq5.fps,
     title="clean render | + neuropil haze | naive footprint-ROI traces (no haze vs +neuropil)")

### Why this notebook does not report $\Delta F/F$

A tempting next step is to normalize each ROI trace as $\Delta F/F = (F(t)-F_0)/F_0$.
In two-photon imaging that is standard: optical sectioning means the light collected
at a cell is *mostly that cell*, so $F_0$ is a meaningful per-cell resting
fluorescence and dividing by it gives a clean, comparable signal.

One-photon miniscope imaging breaks that assumption. As the panels above show, the
light over any footprint is **dominated by background, not by the cell**: the neuropil
pedestal, out-of-focus somata, scattered light, and the static glow we add later
(illumination, vignette, leakage). So the $F$ you measure is mostly background and
$F_0$ is set by that background, which makes a raw $\Delta F/F$ misleading here:

- **Not comparable across cells.** $F_0$ varies across the field with the background,
  so an identical cell sitting on brighter haze reports a *smaller* $\Delta F/F$ than
  one on dimmer haze. The ratio encodes the background's geometry as much as the
  cell's activity.
- **Not stable in time.** The background drifts (population coupling now, and the slow
  photobleaching decline in the next stage), so $F_0$ is not even a fixed number: a
  raw $\Delta F/F$ tracks the drift, not the cell.
- **Wrong order of operations.** The fix is to **demix first** (separate $A\cdot C$
  from $b\cdot f$) to recover each cell's own $C$, and only then normalize if you wish.
  Computing $\Delta F/F$ on the raw ROI normalizes the wrong quantity.

So throughout this notebook the signal of interest is the demixed $C$, judged against
the ground-truth $A$ and $C$ we hold, never a raw-trace $\Delta F/F$.

**Commit the neuropil.** Adds the diffuse background to the committed chain. Idempotent: re-running replaces any prior neuropil so the stage list never accumulates duplicates.

In [ ]:
commit(Neuropil())
print("committed: neuropil (population-driven diffuse background)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

## Stage 5b - Vasculature (the static absorbing landmark)

Stage 5 added the diffuse haze. The other thing almost every cortical recording
shows is **blood vessels**: dark, branching shadows draped across the field. A
vessel is not a light source - haemoglobin *absorbs* both the excitation going in
and the emission coming out - so it is a **multiplicative shadow** cast on
everything optically behind it, darkest under the thick trunks and faint under the
fine capillaries.

Two reasons to model it, pulling in opposite directions:

- **A motion-correction landmark.** The vessels are *fixed in the tissue* and
  high-contrast, and (unlike the cells, whose brightness flickers with activity)
  they look the same in every frame. Registration leans on exactly that: when the
  brain sloshes (Stage 7) the vessels move rigidly with it, so they are the stable
  pattern an algorithm aligns to.
- **An extraction confound.** A vessel crossing a soma darkens it and biases the
  footprint and trace demixing recovers. Turning vasculature up is a knob for
  stress-testing a pipeline.

The vessels are *grown*: each layer sprouts a few branching trees that taper from
thick trunks down to fine capillaries (Murray's law), then is blurred by the
**defocus + scatter at its own depth** - so a shallow layer reads as soft, broad
shadows while a layer near the focal plane stays a crisp thread. Vasculature is
**off by default**; here we switch it on with two layers (a shallow cortical one
and a deeper capillary bed).

In [ ]:
# Tune ONE vessel layer and watch its absorption mask update live -- the real
# vasculature_mask_field, grown on the committed FOV at a fixed seed (so nudging a
# knob refines the same tree, not a new one). The committed focal plane sets the
# per-depth blur, so `depth` sharpens the shadow near focus and softens it far away.
from _anatomy_panels import VasculaturePanel

_focal5b = preview(until="optics").ground_truth.focal_depth_um   # the committed auto-focus depth
_sliders5b = dict(
    depth_um=FloatSlider(min=0.0, max=200.0, step=5.0, value=30.0, description="depth um"),
    n_roots=IntSlider(min=1, max=8, step=1, value=3, description="n vessels"),
    root_radius_um=FloatSlider(min=3.0, max=40.0, step=1.0, value=26.0, description="trunk radius um"),
    opacity=FloatSlider(min=0.10, max=1.0, step=0.05, value=0.85, description="opacity"),
    branch_prob=FloatSlider(min=0.0, max=0.5, step=0.02, value=0.18, description="branchiness"),
    tortuosity_deg=FloatSlider(min=0.0, max=25.0, step=1.0, value=8.0, description="waviness deg"),
)
_vpanel = VasculaturePanel(_sliders5b, acq, _focal5b)
interactive_panel(_sliders5b, _vpanel.draw, _vpanel.canvas)

**Commit the vasculature.** Reads the sliders above, builds one vessel layer from
them, and adds it to the committed chain as a tissue-domain step -- so it rides the
brain motion (Stage 7) rigidly with the cells. Idempotent: re-running replaces any
prior vasculature. (Skip this cell to leave vessels out; the rest of the notebook
runs fine without them.)

In [ ]:
_v5b = {k: s.value for k, s in _sliders5b.items()}
_vasc = Vasculature(enabled=True, layers=[VesselLayer(
    depth_um=_v5b["depth_um"], n_roots=int(_v5b["n_roots"]),
    root_radius_um=_v5b["root_radius_um"], opacity=_v5b["opacity"],
    branch_prob=_v5b["branch_prob"], tortuosity_deg=_v5b["tortuosity_deg"],
)])
commit(_vasc)
print("committed: vasculature (static absorbing vessels; motion landmark + extraction confound)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

In [ ]:
# With those committed settings, render the scope up to the vasculature step: the
# dark vessels hold still while the cells flicker, and the time-average makes the
# static landmark obvious. The ground-truth mask is the exact transmission imposed.
_acq5b = acq.model_copy(update={"duration_s": 12.0})
_rec5b = simulate(Spec(acquisition=_acq5b, seed=SEED, output=Output(save_intermediates=True),
                       steps=list(steps)), until="vasculature")
_movie5b = np.asarray(_rec5b.observed, np.float32)
_mask5b = _rec5b.ground_truth.vasculature_mask
_vmax5b = float(np.percentile(_movie5b, 99.9))
play(_movie5b, fps=_acq5b.fps, height=300, vmax=_vmax5b,
     title="committed scope + vasculature: the dark vessels hold still while the cells flicker")

_fig5b = plt.figure(figsize=(8.6, 4.4))
_g5b = _fig5b.add_gridspec(1, 2, wspace=0.06, left=0.02, right=0.98, top=0.88, bottom=0.04)
_axm = _fig5b.add_subplot(_g5b[0, 0])
_axm.imshow(_mask5b, cmap="gray", vmin=float(_mask5b.min()), vmax=1.0)
_axm.set_xticks([]); _axm.set_yticks([]); _axm.set_title("ground-truth vessel mask (transmission)", fontsize=11)
_axa = _fig5b.add_subplot(_g5b[0, 1])
_axa.imshow(_movie5b.mean(axis=0), cmap=GCAMP, vmin=0, vmax=_vmax5b)
_axa.set_xticks([]); _axa.set_yticks([]); _axa.set_title("time-average: vessels persist, cells average out", fontsize=11)
_fig5b.suptitle("Vasculature is a static, high-contrast landmark -- what registration locks onto", fontsize=12)
show(_fig5b)

## Stage 6 - Photobleaching (and the turnover that fights it)

Fluorophores are not permanent. Every excitation-emission cycle carries a small
chance of a photochemical hit that destroys the molecule for good, so a fluorophore
bleaches in proportion to how much light it emits. Since GCaMP only shines when
calcium-bound, **busier cells bleach faster**, and so do brighter-lit ones, because
turning up the excitation raises both the signal and the damage. Opposing all this,
the cell keeps making fresh protein (**turnover**), pulling expression back toward
full. Per fluorophore pool the intact fraction $B(t)$ obeys

$$\frac{dB}{dt} = \underbrace{\frac{1-B}{\tau_\text{turn}}}_{\text{turnover}} \;-\; \underbrace{q\,I\,c(t)\,B}_{\text{bleaching}}$$

with $I$ the excitation intensity, $c(t)$ the cell's emission (its calcium), $q$ the
per-photon susceptibility, and $\tau_\text{turn}$ the turnover time. The observed
fluorescence is $I\,c(t)\,B(t)$, and composite emits exactly that, leaving the stored
$C$ the clean calcium.

This makes bleaching genuinely biological, not a fixed fade pinned to the clip
length, and it spans two regimes the sandbox below lets you explore:

- **Within a session** (left): brightness decays toward a floor $B^*$ set by how
  hard you drive it. More light or more activity gives a lower floor.
- **Across sessions** (right): with the light off, $B$ recovers as turnover refills
  the pool. If the dark gap is long compared with $\tau_\text{turn}$ the signal
  comes back; if it is too short, the baseline **ratchets down** session by session.

Because the dynamics are physical, the same sandbox is a planning tool: dial in
how hard you drive the indicator and how active the cells are, and read off the
expected fade and recovery for a real experiment. **The dynamics are calibrated to
measured CA1 GCaMP6f bleaching curves** (bleaching linear in excitation; effective
recovery ~5.5 h). Light here is *dimensionless* (1 = a typical continuous level) on
purpose: absolute irradiance depends on the rig, depth, and optics, so we model the
shape of the trade-off, not an absolute power. Turn light up toward ~8 for the bright
intermittent regime, and watch the dark gaps either restore the signal or, when too
short, let it ratchet down.

> **Take the exact numbers with a grain of salt.** This model is *grounded* in real
> miniscope GCaMP6f recordings, but the specific bleach susceptibility and time
> constants are approximate, order-of-magnitude figures, not precise constants. Real
> rates vary widely with indicator variant and expression, excitation power, depth,
> and temperature. Read the *shapes* here (the within-session fade, the across-
> session recovery or ratchet-down) as the lesson, and treat the percentages and the
> ~5.5 h recovery as ballpark values to recalibrate against your own data before
> relying on them quantitatively.


In [ ]:
from _anatomy_panels import BleachingSandbox

# Default = a typical continuous level (light 1) with 60-min sessions every 2 days; turn
# light up to ~8 for the bright intermittent regime. The within/across-session curves use
# real bleaching physics (bleaching_pool/floor/dark_recovery) in _anatomy_panels.
# BleachingSandbox. The numbers are calibrated but approximate -- see the note above.
_sliders6 = dict(
    light=FloatSlider(min=0.05, max=8.0, step=0.05, value=1.0, description="light"),
    activity=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.3, description="activity"),
    turnover_h=FloatSlider(min=1.0, max=72.0, step=0.5, value=5.5, description="recovery (h)"),
    recording_min=FloatSlider(min=5.0, max=180.0, step=5.0, value=60.0, description="recording (min)"),
    gap_days=FloatSlider(min=0.1, max=3.0, step=0.1, value=2.0, description="dark gap (days)"),
)
_bleaching = BleachingSandbox(_sliders6)
interactive_panel(_sliders6, _bleaching.draw, _bleaching.canvas)

**Commit the bleaching.** Adds it to the chain as a cell-domain effect *before*
composite, so each cell emits its clean calcium dimmed by its own envelope, and the
neuropil fades along with the population. At realistic rates the fade over a short
clip is only a few percent (bleaching is slow), so the committed movie barely
changes at this length; the effect compounds over minutes-long recordings and
across sessions.

In [ ]:
commit(Bleaching())   # canonical order places it right after cell_activity, before optics
print("committed: bleaching (per-cell, activity-driven; turnover-limited)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

### Why this matters for a dynamic indicator

There is a subtlety unique to *activity* reporters like GCaMP. A cell only shines
brightly while it is firing, so the fluorophores in the **busiest cells absorb the
most light and bleach the fastest**. As a field photobleaches you therefore see two
things at once: the overall fluorescence drifts down modestly (dominated by the
slowly-bleaching bulk and quiet tissue), but the **calcium-event amplitudes of the
active cells fall faster** - the very signal you care about decays quicker than the
baseline would suggest.

In a real recording this rarely looks like the tissue going dark all at once. What
you see instead is the whole field getting a little dimmer while individual cells
seem to *fade into the background* and switch off one by one. It is tempting to read
that as neurons falling silent or dying. Usually they are doing neither: they have
simply lost working GCaMP, so the same spiking now drives a smaller
calcium-to-fluorescence response and their transients sink toward the noise. The
cells busiest during the recording are the first to fade, because they bleached the
most.

The panels below make this concrete on an intensively-lit recording (light turned up
so the effect shows in tens of minutes instead of hours). The field-average baseline
declines gently; the event amplitudes pooled across active cells (green dots, each a
real transient) fall on a steeper trend (black). The bottom pair freezes a *single
moment of activity* and renders it twice - once fresh, once after a recording's worth
of bleaching - with the Stage-5 diffuse background included. Because the activity and
the cell layout are identical, the only thing that changes is the bleaching itself:
the haze dims a little, and the active cells dim more, fading toward the background.
The practical consequence: a naive $\Delta F/F$ that tracks a slowly drifting baseline
will *underestimate* how much late-session events have shrunk, and demixing must
contend with a signal whose own amplitude is drifting - a thread we pick up in the
final stage.

In [ ]:
from _anatomy_panels import bleaching_fade_figure

# A separate, intensively-lit recording (240 px, 10 fps, 30 min, light up high) so the
# activity-dependent fade is visible in minutes. We build it through optics (no full movie),
# then read the trends + composite the begin/end frames in _anatomy_panels.bleaching_fade_figure
# (it reads clean C, the per-cell bleach envelope B, A_observed, and S straight off the truth).
_acq6b = acq.model_copy(update={"duration_s": 30 * 60.0, "fps": 10.0})
_acq6b = _acq6b.model_copy(update={"image_sensor": acq.image_sensor.model_copy(
    update={"n_px_height": 240, "n_px_width": 240})})
_steps6b = [s for s in steps if s.kind not in ("optics", "composite", "neuropil")]
_steps6b = [s if s.kind != "bleaching" else Bleaching(excitation_intensity=10.0)
            for s in _steps6b] + [CellOptics()]
_rec6b = simulate(Spec(acquisition=_acq6b, seed=SEED, output=Output(), steps=_steps6b), until="optics")
_np6 = next(s for s in steps if s.kind == "neuropil")
_fig6b = bleaching_fade_figure(_rec6b.ground_truth, _np6, _acq6b, seed=SEED)
show(_fig6b)

## Stage 7 - Brain motion (the brain sloshes inside the skull)

The Miniscope is bolted rigidly to the skull, but the brain is not bolted to
anything. It has the consistency of soft jello, tethered to the skull by membranes
and cerebrospinal fluid, and it **sloshes** as the animal moves. Through the scope
this shows up as the whole field of view drifting and jittering by roughly
**+/- 10 um** around a rest position. (Different optical implants - a GRIN lens, a
cranial window, a prism - change how tightly the tissue is coupled to the skull and
so change this motion; we mention it but do not model it.)

Most of that motion is **broadband sloshing** - irregular, drift-like, driven by
respiration, heartbeat, and the constant small jostles of behavior. But woven through
it is a **rhythm** set by **locomotion**: mice and rats run with a stride frequency of
about **6 to 8 Hz**, and every footfall sends a small impulse up through the body and
skull into the brain. So brain motion is neither a pure random walk nor a clean
oscillation - it is mostly noise, with a stride rhythm riding within it.

### How we model it

We treat the brain as exactly what it is: a **damped mass on an elastic tether**.
Per axis,

$$\ddot{x} + 2\zeta\omega_0\,\dot{x} + \omega_0^2\,x \;=\; a_\text{drive}(t)$$

a driven damped harmonic oscillator. The restoring term $\omega_0^2 x$ is the tether
pulling the tissue back toward rest; the damping $2\zeta\omega_0\dot{x}$ is jello
bleeding off energy; the inertia means the tissue cannot teleport or reverse
direction instantly. Two things drive it: broadband **sloshing** noise on both axes
(the bulk of the motion), plus a smaller always-on **locomotion** acceleration at the
stride frequency (`locomotion_freq_hz`, default 7 Hz) along the dominant bobbing axis.
`locomotion_fraction` sets how much of the motion is rhythm versus noise (default 0.25
- noise-dominated).

That gets us the behaviour we want for free. The motion stays **bounded** because the
tether keeps pulling it back - no artificial clamp is doing the work. It has
**inertia**, so it is smooth, never a jagged random walk. The spectrum is **red** -
most of the power is at low frequencies (slow drift, walking, head movement) - with
the stride rhythm a modest **bump near 7 Hz**, not a dominant tone. `motion_amplitude_um`
sets the *extreme* excursion (default ~10 um, the 99th percentile); most frames move
considerably less, clustered near the rest position. As with the spikes in Stage 3,
we integrate the oscillator finely and then average down to the camera frame rate,
because the camera exposure integrates over time.

We apply the result as a single **rigid translation** of the whole field - which is
what image registration later estimates and corrects. The tissue's true sloshing is
mildly *non-rigid* and can carry a little rotation; those are real but beyond v1.

### Why this puts a floor under your frame rate

Here is the practical payoff, and it is worth internalizing *before* you ever record.
The brain is oscillating at ~7 Hz; your camera samples it at some frame rate. Two
regimes follow:

- **Motion *between* frames.** Sample fast enough and each frame is a sharp snapshot;
  the brain has merely moved a little since the previous one. That is a **rigid shift
  you can measure and correct** afterwards (registration). This is the good case, and
  it is the only motion we model.
- **Motion *within* a frame.** If a single exposure lasts long enough that the brain
  moves appreciably *while the shutter is open*, the frame itself is **smeared** -
  motion blur baked into the pixels. No registration can undo that; the information
  is simply gone.

So locomotion at 6 to 8 Hz sets a **lower bound on your frame rate**: sample well
above the stride frequency (and expose for well under one stride) so motion lands
between frames, not within them. We model the between-frame shift and deliberately do
**not** simulate within-frame blur - so if you starve the frame rate this simulator
will still hand you crisp frames, while a real scope would not. Treat the frame rate
as something *you* are responsible for keeping high enough. The UCLA V4 we committed
runs at 20 fps; against a 7 Hz stride that is just under three samples per cycle -
workable, but only just: in the spectrum below the stride bump sits close to the
Nyquist limit, and any slower would start folding it back as a phantom slow drift.

In [ ]:
from minisim.steps import physical_brain_motion
from _anatomy_panels import motion_diagnostics_figure

# physical_brain_motion is exactly the generator simulate() uses for the committed step.
# We build two trajectories from the BrainMotion defaults: a fine ~continuous one for the
# trajectory panel, and a long (~8 min) one at the camera rate for the spectrum + shift
# cloud. The welch spectrum + 2-D histogram + 3-panel layout live in _anatomy_panels.
_mot7 = BrainMotion()
_px_um = acq.pixel_size_um


def _traj(n, fps, seed):
    return physical_brain_motion(
        n, fps=fps, locomotion_freq_hz=_mot7.locomotion_freq_hz,
        resonance_freq_hz=_mot7.resonance_freq_hz, damping_ratio=_mot7.damping_ratio,
        locomotion_fraction=_mot7.locomotion_fraction, locomotion_axis=0,
        amplitude_px=acq.um_to_px(_mot7.motion_amplitude_um),
        max_px=acq.um_to_px(_mot7.max_shift_um), rng=np.random.default_rng(seed)) * _px_um


_und = _traj(int(5.0 * 200.0), 200.0, SEED)             # fine, near-continuous
_long = _traj(int(8 * 60 * acq.fps), acq.fps, SEED)     # long, at the camera rate
_fig7 = motion_diagnostics_figure(_und, 200.0, _long, acq.fps,
                                  amplitude_um=_mot7.motion_amplitude_um,
                                  locomotion_freq_hz=_mot7.locomotion_freq_hz)
show(_fig7)

Read left to right: the trajectory is **mostly irregular sloshing**, with the stride
rhythm too subtle to pick out by eye on the locomotion axis (the cross axis only
drifts); the spectrum is **red** - most power at low frequencies (slow drift, walking,
head movement) - with the stride rhythm a modest **bump near 7 Hz**, which still sits
close to the 20 fps Nyquist limit; and the per-frame shift distribution is a
**Gaussian-like cloud** centred on the rest position, thinning out toward the dashed
+/- 10 um extreme. That last panel is the honest summary: the brain spends most of the
recording close to home and only occasionally swings to the edge. Now watch it in the
movie, and see what even this modest motion costs the image.

In [ ]:
# Render the committed scope over a short window with vs without brain motion on a
# modest FOV. We drop the neuropil haze so the cost lands on the cells themselves: watch
# the field jitter in the video, then the time-average makes it concrete -- each cell
# smears by roughly the RMS motion, exactly what later registration undoes.
_acq7 = acq.model_copy(update={"duration_s": 10.0,
    "image_sensor": acq.image_sensor.model_copy(update={"n_px_height": 220, "n_px_width": 220})})
_cells7 = [s for s in steps if s.kind != "neuropil"]   # haze-free, so the streak is what you see
# Motion must be in BOTH specs: it sizes the tissue margin, and cells are placed across the
# margined canvas -- so a no-motion spec would render a DIFFERENT field (more cells, shifted
# positions), not the same field held still. We keep motion in both and read the un-shifted
# view at cells_only, so the ONLY difference between the panels is the per-frame shift. The
# amplitude is exaggerated here (the committed ~10 um excursion smears by only its RMS, a few
# um) so the per-cell streak is unmistakable.
_mot7 = BrainMotion(motion_amplitude_um=30.0, max_shift_um=40.0)
_steps7 = _cells7 + [_mot7]
def _run7(until):
    return np.asarray(simulate(Spec(acquisition=_acq7, seed=SEED, output=Output(), steps=_steps7),
                               until=until).observed, float)
_still7 = _run7("cells_only")    # same field, un-shifted (rest position)
_move7 = _run7("brain_motion")   # same field, shifted + cropped per frame
play(_move7, fps=_acq7.fps, height=300,
     title="committed scope + brain motion (exaggerated): the whole field jitters at the stride rhythm")

_c7 = slice(50, 170)   # zoom the center so the per-cell streak is legible
_sm7, _mm7 = _still7.mean(axis=0)[_c7, _c7], _move7.mean(axis=0)[_c7, _c7]
_vmx7 = float(np.percentile(_sm7, 99.8))
_fig7b = plt.figure(figsize=(8.0, 4.3))
_g7b = _fig7b.add_gridspec(1, 2, wspace=0.06, left=0.02, right=0.98, top=0.88, bottom=0.04)
for _j, (_img, _ttl) in enumerate([(_sm7, "time-average, no motion (crisp)"),
                                   (_mm7, "time-average, with motion (smeared)")]):
    _ax = _fig7b.add_subplot(_g7b[0, _j])
    _ax.imshow(_img, cmap=GCAMP, vmin=0, vmax=_vmx7); _ax.set_xticks([]); _ax.set_yticks([])
    _ax.set_title(_ttl, fontsize=11)
_fig7b.suptitle("Uncorrected motion smears the time-average (motion exaggerated for visibility)", fontsize=12)
show(_fig7b)

**Commit the brain motion.** Adds it to the chain as a motion-domain step - it sits after the tissue stages (it moves the brain-frame content: cells and neuropil) and before the sensor-frame stages still to come. `simulate()` sizes the tissue margin automatically from `max_shift_um`, so real off-FOV tissue moves into view rather than a fabricated fill. Idempotent: re-running replaces any prior motion step.

In [ ]:
commit(BrainMotion())   # motion-domain: canonical order slots it after tissue, before sensor
print("committed: brain_motion (driven damped oscillator; ~7 Hz locomotion, +/-10 um)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

## Stage 8 - Uneven illumination and vignetting (the static brightness falloff)

Look at almost any miniscope recording and the **center is brighter than the edges**.
This is two separate physical effects that happen to look the same in the image:

1. **Uneven illumination (excitation).** A single excitation LED lights the tissue
   brightest at the center and dimmer toward the edges, so peripheral cells are
   excited less and **fluoresce less** to begin with. Usually a broad, gentle rolloff.
2. **Vignetting (emission / return path).** On the way back, the physical return path
   trims light rays toward the field edges (aperture and relay clipping, plus poorer
   off-axis optical performance), so the corners **collect less light** regardless of
   how brightly the tissue was lit. Usually a sharper rolloff near the rim.

Both are radial, both multiply the image, and **both are bolted to the scope** - so,
unlike the brain motion of Stage 7, they are *static*: the bright center stays put
frame after frame while the tissue jiggles underneath. We apply each as its own
multiplicative field after the motion crop, and record both to ground truth.

### Why it matters

- **It shrinks your usable FOV.** Edge cells get less excitation *and* less of their
  emission collected, so they reach the sensor with far fewer photons - lower SNR, and
  some drop below detection entirely. Your *usable* field of view is smaller than your
  *physical* one. (Note this is a **photon-budget** problem, not something $\Delta F/F$
  rescues: in 1-photon imaging the baseline $F$ is always contaminated by background,
  so we don't lean on raw $\Delta F/F$ - real recovery comes from demixing, and the
  falloff sets the photon budget and the background that demixing must separate.)
- **It can sabotage motion correction.** The falloff is a strong, *stationary*,
  low-frequency pattern. Registration aligns frames by matching spatial structure, and
  it can happily lock onto this fixed bright rim instead of the faint *moving* brain
  features - and then under-correct the real motion. The fix in practice: flat-field
  or high-pass the movie *before* registering, so the moving features dominate what the
  aligner sees.

### The one way the two differ

On the image they are nearly interchangeable, but the **illumination profile drives
photobleaching** while the vignette does not: more excitation light at the center means
the central cells bleach faster (the Stage-6 effect, now spatially patterned). So when
we commit the illumination profile, `Bleaching` scales each cell's excitation dose by
the illumination at its rest position. The vignette is pure collection loss and touches
no bleaching.

*(Off-axis **blur** - coma, astigmatism, the field-dependent PSF - is a real edge
effect too, but it is aberration, not dimming, and gets its own stage later. We already
model one piece of it, field curvature, back in the optics stage.)*

Each falloff has two knobs. **corner** is its brightness at the farthest corner of the
FOV, relative to 1 at the center (so 0.5 means the corner is half as bright). **rolloff**
is how sharply it gets there: 1 falls off linearly with radius, while higher values keep
the center nearly flat and then drop steeply toward the rim. All three plots span the
**full FOV**, so each profile reaches its corner value at the far edge.

Drag the four sliders and watch the edge cells slide below the detection floor. This uses
the **scope, optics, and cell layout you committed** in the earlier stages, imaged over a
short window, so the cells that wink out are the ones your actual recording would lose.
Only the fields and the per-cell SNR recompute as you drag - the composite underneath is
fixed - and the floor uses a representative sensor (committed in the next stage).

In [ ]:
from _anatomy_panels import IlluminationPanel

# Render the COMMITTED scope ONCE (full FOV, optics/sensor/cell layout locked in earlier);
# then only the two cheap falloff fields + the per-cell SNR recompute as you drag. The
# detection floor uses a representative sensor. Render + figure: _anatomy_panels.IlluminationPanel.
_sliders8 = dict(
    illum_corner=FloatSlider(min=0.2, max=1.0, step=0.05, value=0.7, description="illum corner"),
    illum_rolloff=FloatSlider(min=1.0, max=4.0, step=0.5, value=2.0, description="illum rolloff"),
    vig_corner=FloatSlider(min=0.2, max=1.0, step=0.05, value=0.5, description="vignette corner"),
    vig_rolloff=FloatSlider(min=1.0, max=6.0, step=0.5, value=4.0, description="vignette rolloff"),
)
_illum = IlluminationPanel(_sliders8, acq, steps, SEED)
interactive_panel(_sliders8, _illum.draw, _illum.canvas)

**Commit the illumination profile and vignette.** Both are sensor-frame fields, so they sit after `brain_motion` (they are fixed to the scope, not the brain). The illumination profile also feeds `Bleaching` - center cells bleach faster - which `simulate()` wires up automatically. Idempotent: re-running replaces any prior copies.

In [ ]:
commit(IlluminationProfile(),   # excitation falloff (also drives bleaching at the center)
       Vignette())              # emission / return-path collection loss
print("committed: illumination_profile + vignette (static, scope-fixed brightness falloff)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

## Stage 9 - The image sensor (clean intensity becomes raw, noisy counts)

Everything so far has been a clean, continuous *intensity* field - real numbers with no
measurement noise. A real recording never gives you that. The **image sensor** is where
the physics finally turns into a photo: incident light becomes a noisy stream of
**integer ADC counts**, and three things appear that simply did not exist upstream.

**1. Leakage - stray excitation light on the detector.** Before we even digitize, a
static background sits on top of the image. A miniscope has almost no room for a
well-behaved thin-film emission filter, so its cutoff is imperfect: a little of the
bright **excitation** light slips through, and more of it bounces around inside the
cramped housing and finds its way to the sensor. This is *excitation* light hitting the
detector, **not** fluorescence - so it does not bleach and does not move with the brain.
We model it as an additive, scope-fixed glow, brightest near center where the excitation
and the internal reflections concentrate.

> **"Glow" is the whole low-frequency background, not just leakage.** The slowly varying
> haze you see - and what minian's *glow removal* estimates and subtracts - is the
> **combined** result of uneven illumination and vignette (the multiplicative shapes from
> Stage 8) *plus* this additive leakage. Leakage is one contributor; the removal tackles
> the bundle, because all three are smooth and static while the cells are sharp and moving.

**2. Shot noise - the irreducible floor.** Photon arrival is Poisson, and detection keeps
it Poisson, so the electrons a pixel collects have standard deviation $\sqrt{N}$. Brighter
pixels are *relatively* cleaner ($\mathrm{SNR}\sim\sqrt{N}$), which is why **the photon
budget is the master knob**: `photons_per_unit` is your exposure / LED power. Turn it down
and read noise dominates (dim cells vanish into the floor); turn it up and you become
shot-limited - the best you can do - until the brightest pixels **saturate**.

**3. Read noise, quantization, clipping.** The readout adds a fixed Gaussian noise (a few
electrons), the ADC **floors** to integers, and an **8-bit** converter has only 256 codes:
dim transients can fall *between* codes, and anything past the top code is **clipped** -
saturation, and it is irreversible. (Pixel pitch, QE, read noise, gain, and bit depth are
sensor hardware on `Acquisition.image_sensor`; the exposure scale is the one scene-side
knob, on the `Sensor` step.)

### Why it matters

This is where **SNR becomes real**. Nowhere upstream did we set a signal-to-noise ratio by
hand; it *emerges* here, from the photon budget against the shot + read floor. That floor is
exactly what the `detectable` flag and the auto-focus *yield* are judged against - so
**committing the sensor is what makes the focus we built earlier go live**: with a noise
floor finally defined, `"auto"` stops minimizing average blur and starts **maximizing the
count of cells that clear the floor** (re-run the optics cell after committing and watch the
resolved focus move).

Drag the knobs: drop the **exposure** and dim cells sink into the read floor (the SNR panel
empties); raise it and the histogram climbs toward the bit-depth ceiling until pixels
**saturate**; raise the **leakage glow** and a contaminating central haze swells over the
image - the low-frequency background that glow-removal must strip before any extracted trace
is trustworthy (it shows in the picture and the histogram, not the per-cell SNR, which is
judged against each cell's own baseline). This uses the **committed scope, optics, cell
layout, and illumination/vignette**, so the counts here are the ones your real recording
produces.

In [ ]:
from _anatomy_panels import SensorPanel

# Render the COMMITTED chain ONCE to a clean intensity frame (illumination + vignette folded
# in); then only the additive leakage glow + digitization recompute as you drag. Render +
# figure (raw counts, count histogram, per-cell SNR vs radius): _anatomy_panels.SensorPanel.
_sliders9 = dict(
    photons=FloatSlider(min=20.0, max=1500.0, step=20.0, value=300.0, description="exposure (ph)"),
    read_noise=FloatSlider(min=0.0, max=10.0, step=0.5, value=2.0, description="read noise (e-)"),
    bit_depth=IntSlider(min=6, max=12, value=8, description="bit depth"),
    leak=FloatSlider(min=0.0, max=0.5, step=0.025, value=0.1, description="leakage glow"),
)
_sensor = SensorPanel(_sliders9, acq, steps, SEED)
interactive_panel(_sliders9, _sensor.draw, _sensor.canvas)

**Commit leakage and the sensor.** Leakage is additive, so it goes on *before* digitization; the sensor is the final step that turns intensity into integer counts. Both are sensor-frame (after `brain_motion` and the falloff fields). Committing the sensor also hands `simulate()` the noise floor it needs, so `detectable` and the `"auto"` yield-focus go live - re-run the optics cell and the resolved focus is no longer the plain median. Idempotent.

In [ ]:
commit(Leakage(),                          # additive stray-excitation glow (static, does not bleach)
       Sensor(photons_per_unit=300.0))     # photons -> e- -> shot + read -> quantize -> clipped counts
print("committed: leakage + sensor (the movie is now raw integer counts)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

### Watch it - and take it with you

The recording is one `simulate(spec)` away. Below we (1) play a short clip **inline** so
you can see the synthetic data move, and (2) stream the **full** recording to a grayscale
**AVI** on disk. The write is *chunked* - frames are rendered, digitized, and written a few
at a time - so the whole movie never has to fit in memory, and the very same call scales to
recordings far longer than this one. A progress bar tracks the write.

> The raw counts are honestly dim (you saw why in Stage 9), so the inline preview is
> contrast-stretched **for display only**; the file on disk keeps the true counts. Want a
> longer recording? Raise `duration_s` on the committed scope and re-run - `simulate_video`
> streams it without growing memory.

In [ ]:
from pathlib import Path

from minisim import simulate_video

full_spec = Spec(acquisition=acq, seed=SEED, steps=steps)   # the entire committed chain

# (1) short inline preview: simulate ~8 s and play it (contrast-stretched for display).
_clip = simulate(Spec(acquisition=acq.model_copy(update={"duration_s": 8.0}),
                      seed=SEED, steps=steps)).observed
_vmax = float(np.percentile(_clip, 99.7)) or 1.0
mediapy.show_video(np.clip(_clip / _vmax, 0.0, 1.0), fps=acq.fps,
                   title=f"first {_clip.shape[0] / acq.fps:.0f} s (contrast-stretched for display)")

# (2) full recording -> grayscale AVI, streamed to disk: bounded memory + a progress bar.
out = simulate_video(full_spec, Path("01_anatomy_recording.avi"))
print(f"wrote {out.resolve()}  ({out.stat().st_size / 1e6:.1f} MB, "
      f"{acq.n_frames} frames @ {acq.fps:.0f} fps)")

---
## The forward model is complete

From a fluorescent protein to raw pixels, every stage was driven by a physical cause:
**place_neurons -> cell_activity -> bleaching -> optics -> composite -> neuropil ->
brain_motion -> illumination_profile -> vignette -> leakage -> sensor**. Nothing here was
shaped to suit an analysis algorithm; it is the biology and the instrument, and each stage
carried its own **ground truth** alongside the movie.

That is the whole point. A real analysis pipeline - motion correction, glow removal, demixing
the overlapping footprints, deconvolving the traces - is an attempt to *invert* this chain.
Because you kept the ground truth (true footprints `A`, true traces `C`, the resolved focus,
the static fields), you can grade exactly how much of the biology any method actually recovers,
and *why* it fails where it does. That is where the next notebooks go.